In [0]:
from pyspark.sql import functions as F

spark.sql('CREATE SCHEMA IF NOT EXISTS silver')

In [0]:
customers = spark.table('bronze.customers')
customers.show(5, truncate=False)

In [0]:
customers.select('country').distinct().show()

In [0]:
customers_validation = (
    customers.withColumn(
        '_dq_status',
        F.when(F.col('customer_id').isNull(), F.lit('FAIL'))
        .when(F.col('email').isNull() | ~F.col('email').rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), F.lit('FAIL'))
        .when(~F.col('country').isin('Germany', 'China','Australia','UK','Malaysia','Japan','India'), F.lit('FAIL'))
        .otherwise(F.lit('PASS'))
    )
)

customers_validation = (
    customers_validation.withColumn(
        '_dq_reason',
        F.when(F.col('customer_id').isNull(), F.lit('MISSING_CUSTOMER_ID'))
        .when(F.col('email').isNull(), F.lit('MISSING_EMAIL'))
        .when(~F.col('email').rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), F.lit('INVALID_EMAIL'))
        .when(~F.col('country').isin('Germany', 'China','Australia','UK','Malaysia','Japan','India'), F.lit('INVALID_COUNTRY'))
        .otherwise(F.lit(''))
    )
)

customers_validation.groupBy('_dq_status').count().show()

In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS quarantine')

In [0]:
customers_silver = (
    customers_validation
    .filter('_dq_status == "PASS"')
    .drop('_dq_status','_dq_reason')
)

customers_quarantine = (
    customers_validation
    .filter('_dq_status == "FAIL"')
)
customers_silver.write.format('delta').mode('overwrite').saveAsTable('silver.customers')
customers_quarantine.write.format('delta').mode('overwrite').saveAsTable('quarantine.customers')

In [0]:
display(customers_quarantine)